# Structured outputs

## Lesson goals
* Recognise when to reach for structured outputs vs. older techniques
* Pass a JSON schema through `output_config.format` so Claude's response is guaranteed schema-valid
* Know which techniques are incompatible with structured outputs and when to fall back

## Why structured outputs

The earlier lessons covered two ways to coax structured JSON out of Claude:

1. **Prefill plus stop sequence** — start the assistant turn with `` ```json `` and stop on `` ``` ``.
2. **Tool use** — describe a tool's input schema and treat the resulting `tool_use` block as the payload.

Both work, but they leave the actual JSON validity up to the model. You still parse the response, you still retry on `json.loads` errors, and you still hand-write the closing brace logic.

`output_config.format` uses *constrained decoding* — the API only generates tokens that keep the response valid against the schema you supply. There is no parse step that can fail, no retry loop, and no prefill is needed.

Structured outputs are GA on Opus 4.7, Opus 4.6, Sonnet 4.6, Sonnet 4.5, Opus 4.5, and Haiku 4.5. See the [structured outputs reference](https://platform.claude.com/docs/en/build-with-claude/structured-outputs) for the full list.


As always, start by importing the SDK and creating a client:

In [ ]:
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()
client = Anthropic()

## Extracting a contact card

Suppose you have a free-form bio and want to pull out a structured contact record. Define the JSON schema you want back:

In [ ]:
contact_schema = {
    "type": "object",
    "properties": {
        "name": {"type": "string"},
        "email": {"type": "string"},
        "company": {"type": "string"},
        "role": {"type": "string"},
    },
    "required": ["name", "email", "company", "role"],
    "additionalProperties": False,
}

Pass it through `output_config.format` and read the response straight out as JSON — no prefill, no stop sequence, no retry loop:

In [ ]:
import json

bio = (
    "Hi, I'm Alex Rivera (alex.rivera@northwind.example), "
    "Staff Software Engineer at Northwind Robotics. "
    "I lead the perception team there."
)

response = client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=512,
    messages=[
        {"role": "user", "content": f"Extract the contact information from this bio:\n\n{bio}"},
    ],
    output_config={
        "format": {
            "type": "json_schema",
            "schema": contact_schema,
        },
    },
)

# The first content block is the structured payload — guaranteed schema-valid.
contact = json.loads(response.content[0].text)
contact

Two things to notice:

* You did **not** prefill the assistant turn. Adding a prefill is incompatible with structured outputs — the API will reject the request.
* `json.loads` is here only to turn the text into a Python `dict`. It cannot raise `JSONDecodeError`, because the API only emitted tokens that keep the output schema-valid.


## Comparison: the prefill + stop-sequence technique

For reference, the older pattern looks like this. Use it when you need structured-ish output from a model that doesn't support structured outputs yet, or when you need a payload shape that doesn't fit a JSON schema (e.g. interleaved prose and JSON):

In [ ]:
legacy_response = client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=512,
    messages=[
        {"role": "user", "content": f"Extract the contact information from this bio:\n\n{bio}"},
        {"role": "assistant", "content": "```json\n"},
    ],
    stop_sequences=["```"],
)

raw = legacy_response.content[0].text  # may or may not be valid JSON

try:
    contact_legacy = json.loads(raw)
except json.JSONDecodeError:
    # The retry / repair logic that structured outputs makes unnecessary.
    contact_legacy = None

contact_legacy

When to pick which:

| Need | Use |
| --- | --- |
| Strict JSON conforming to a known schema | `output_config.format` (structured outputs) |
| Mixed prose + a JSON snippet | prefill + stop sequence |
| Function/tool invocation with arguments | tool use (`tools=[...]`) |
| Model without structured-outputs support | prefill + stop sequence, or upgrade the model |

Structured outputs are also incompatible with assistant prefill and with `stop_sequences` that would land mid-payload, so combine them carefully.


## Exercise

Use structured outputs to extract a **list** of meetings from a short email digest.

Schema sketch:

```json
{
  "type": "object",
  "properties": {
    "meetings": {
      "type": "array",
      "items": {
        "type": "object",
        "properties": {
          "title":     {"type": "string"},
          "starts_at": {"type": "string"},
          "attendees": {"type": "array", "items": {"type": "string"}}
        },
        "required": ["title", "starts_at", "attendees"]
      }
    }
  },
  "required": ["meetings"]
}
```

Sample digest:

> Reminder: standup with the Perception team at 10:00 on Tuesday, then a 1:1 with Jordan at 14:30. Friday's release readiness review is at 16:00 with Sam, Priya and Lin.

Expected output: a `meetings` array with three entries, each with `title`, `starts_at`, and `attendees`. Verify that parsing the response with `json.loads` never raises.
